# oscNext L4 — uçtan uca (LightGBM)

L3 `.i3` dosyalarından L4 sınıflandırıcılarına kadar tüm süreç.

**Yöntem teknik notunkiyle aynı:** LightGBM, Tablo 10 hiperparametreleri.
(Proje bir süre pybdt/AdaBoost denedi; aynı veri üzerinde ölçüldü ve %99
gürültü reddinde %65.8'de kaldı, LightGBM %95.9 verdi. Ayrıntı `CLAUDE.md`.)

## Çalıştırma ortamı

Kernel, IceTray build'inin `env-shell.sh`'i içinden başlatılmış python
olmalı — yoksa `icecube.*` import edilemez. Bkz. `README.md`.

## Ağır iş `.py` dosyalarında

Notebook ince bir arayüz: işleme `process_L4.py`, veri okuma `l4_data.py`,
sürücü `l4_run.py`, eğitim `train_L4_classifier.py`. Mantık burada
tekrarlanmıyor ki tek implementasyon kalsın.


## 0. Konfigürasyon ve ortam kontrolü

`lightgbm` ve `tables` zorunlu. `pandas` **kullanılmıyor** — IceTray
ortamında bulunmayabilir.


In [ ]:
# l4_run.py / l4_data.py degistiginde kernel'i yeniden baslatmaya gerek
# kalmasin diye: modul dosyasi degisince otomatik yeniden yuklenir.
# (Bu olmadan `git pull` sonrasi "cannot import name ... from l4_data"
#  gibi hatalar alirsin -- dosyada var ama hafizadaki modul eski.)
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    print("autoreload  ACIK  (l4_*.py degisiklikleri otomatik gelir)")
except Exception:
    pass

import os, sys, glob, json, shlex, subprocess, time
import numpy as np

# --- zorunlu: lightgbm ---
try:
    import lightgbm as lgb
    print("lightgbm     OK  ", lgb.__version__)
except ImportError as e:
    raise SystemExit(
        "lightgbm import edilemedi (%s).\n"
        "Egitim bu olmadan yapilamaz.  IceTray ortamindan deneyin:\n"
        "  ./setup_env.sh run python -c 'import lightgbm'" % e)

# --- zorunlu: pytables ---
try:
    import tables
    print("tables       OK  ", tables.__version__)
except ImportError:
    raise SystemExit("pytables yok -- HDF5 okunamaz.")

# --- opsiyonel ---
_NOTE = {"matplotlib": "grafik cizilemez",
         "simweights": "CORSIKA agirligi yaklasik olur",
         "ipywidgets": "ilerleme cubugu ASCII'ye duser"}
for _n in _NOTE:
    try:
        _m = __import__(_n)
        print("%-12s OK   %s" % (_n, getattr(_m, "__version__", "")))
    except ImportError:
        print("%-12s YOK  (%s)" % (_n, _NOTE[_n]))

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

In [ ]:
# ---------------------------------------------------------------------------
# YOLLAR
# ---------------------------------------------------------------------------
# Scriptlerin (process_L4.py, train_L4_classifier.py, l4_*.py) bulundugu dizin
L4_CODE_DIR = os.environ.get("OSCNEXT_L4_CODE", ".")
if os.path.abspath(L4_CODE_DIR) not in sys.path:
    sys.path.insert(0, os.path.abspath(L4_CODE_DIR))

# Ciktilar.  HDF5 ONLARCA GB olabilir -- home dizininde kota varsa
# OSCNEXT_OUT_ROOT'u /data/user/$USER/... altina alin.
OUTPUT_ROOT = os.environ.get(
    "OSCNEXT_OUT_ROOT", os.path.join(os.path.abspath(L4_CODE_DIR), "L4_output"))

HDF_BASE  = os.path.join(OUTPUT_ROOT, "hdf5")     # process_L4.py ciktisi
DS_BASE   = os.path.join(OUTPUT_ROOT, "ds")       # .npz egitim setleri
MODEL_DIR = os.path.join(OUTPUT_ROOT, "models")   # .txt + .json + grafikler

for _d in (HDF_BASE, DS_BASE, MODEL_DIR):
    os.makedirs(_d, exist_ok=True)

PROCESS_PY = os.path.join(L4_CODE_DIR, "process_L4.py")
TRAIN_PY   = os.path.join(L4_CODE_DIR, "train_L4_classifier.py")

RNG_SEED = 12345
rng = np.random.default_rng(RNG_SEED)

_st = os.statvfs(OUTPUT_ROOT)
print("Cikti koku : %s" % OUTPUT_ROOT)
print("Bos disk   : %.1f GB" % (_st.f_bavail * _st.f_frsize / 1e9))
for _p in (PROCESS_PY, TRAIN_PY):
    print("%-24s %s" % (os.path.basename(_p),
                        "var" if os.path.exists(_p) else "YOK!"))

## 1. L3 → L4 işleme

`process_L4.py`'yi her örnek için çalıştırır. **Uzun sürer**, bir kez yapılır —
HDF5'ler üretildikten sonra 2. bölümden devam edebilirsin.

`--apply-cut` **kullanılmıyor**: modeller eğitilmeden önce tüm olaylar book
edilmeli, yoksa eğitim setini kesmiş oluruz.

In [ ]:
GCD = "/cvmfs/icecube.opensciencegrid.org/data/GCD/GeoCalibDetectorStatus_IC86.All_Pass3.i3.gz"

# pass3 uretimi.  nutau ve gercek dedektor verisi bu uretimde YOK.
SAMPLES = {
    "nue":     dict(l3="/data/ana/LE/oscNext/pass3/genie/level3/23800/*.i3.zst",
                    flags=["--mc", "--genie"], kind="signal"),
    "numu":    dict(l3="/data/ana/LE/oscNext/pass3/genie/level3/23799/*.i3.zst",
                    flags=["--mc", "--genie"], kind="signal"),
    "corsika": dict(l3="/data/ana/LE/oscNext/pass3/corsika/level3/23694/*.i3.zst",
                    flags=["--corsika"], kind="muon_bg"),
    "noise":   dict(l3="/data/ana/LE/oscNext/pass3/noise/level3/23813/*.i3.zst",
                    flags=["--noise"], kind="noise_bg"),
}

for name, cfg in SAMPLES.items():
    cfg["hdf5"] = os.path.join(HDF_BASE, name, "L4_%s.hdf5" % name)
    cfg["n_l3_files"] = len(glob.glob(cfg["l3"]))
    print("%-8s %-10s %6d L3 dosyasi" % (name, cfg["kind"], cfg["n_l3_files"]))

In [ ]:
from l4_run import configure_runner, run_process, run_all
configure_runner(SAMPLES, PROCESS_PY, GCD)

### Önce smoke test

Tam üretime geçmeden tek dosyada 200 frame işleyip zincirin çalıştığını
doğrula.

> **`--n` FRAME sayar, olay değil.** Akışta G/C/D, Q ve P frame'leri var;
> P frame'lerin de ancak bir kısmı `InIceSplit`'e uyup L3 kesimini geçiyor.
> 200 frame → ~60 olay normal. Çıktı kademeyi gösteriyor.

In [ ]:
smoke = run_process("nue", n_frames=200)

### Tam üretim

`chunk_files=10` → her 10 L3 dosyası ayrı bir parça (`L4_nue_part000.hdf5`, …).

- **Gerçek yüzde ve ETA** — parça sayısı baştan belli.
- **Kaldığı yerden devam** — tamamlanan parçalar atlanır, çökme halinde
  sadece o parça kaybolur.

Bozuk `.i3.zst` dosyaları `--scan quick` (varsayılan) ile eleniyor; tray yine
patlarsa `--retries` o dosyayı atıp devam ediyor.

In [ ]:
# jobs>1 -> her ornek N paralel surecte islenir (en buyuk hizlanma).
#           cobalt PAYLASILAN makine: 8 makul, 64 degil.
# skip_optional=True -> I3TensorOfInertia ve separation_in_cogs
#           hesaplanmaz.  Ikisi de Tablo 11/12'de YOK, yani BDT girdisi
#           degil.  Sonradan lazim olurlarsa yeniden isleme gerekir.
results = run_all(jobs=8, chunk_files=10, skip_optional=True)

## 2. Booking doğrulaması

**İşleme bittikten sonra ilk iş bu.** Bir tray modülü sessizce başarısız
olursa tablo hiç yazılmaz; bunu üç bölüm sonra "bu değişken neden hep NaN"
diye keşfetmek yerine burada yakala.

In [ ]:
from l4_data import dump_tables, find_hdf5

# Sabit "_smoke.hdf5" yazmiyoruz: smoke test calistirilmadiysa o dosya YOK
# ama uretim ciktisi var.  find_hdf5 hangisi varsa onu bulur.
H5 = find_hdf5("nue", SAMPLES)
TABLES = dump_tables(H5) if H5 else {}

In [ ]:
# Belirli bir tabloya yakindan bakmak icin, orn. iLineFit kolon adi
# (REGISTRY/ALTS'in dogru kolonu sectigini dogrulamak icin):
# dump_tables(H5, only=["iLineFit"])

## 3. Feature registry ve tutarlılık

`REGISTRY` her BDT değişkenini `(HDF5 tablosu, kolon)` çiftine bağlar.
`ALTS` isim varyasyonlarını çözer — meta-proje/pass sürümüne göre kolon
adları değişiyor, dosyada **gerçekten hangisi varsa** o kullanılır.

`check_feature_map()` `REGISTRY` ile `l4_classifier_module.FEATURE_MAP`
çakışıyor mu diye bakar. Eğitimde bir kolon, frame'e uygularken başka bir
kolon okunursa model **hata fırlatmadan** saçmalar — bu yüzden kodla
kontrol ediliyor (AST ile okuyor, icetray gerekmiyor).

In [ ]:
import l4_data
from l4_data import (REGISTRY, ALTS, AUX, NOISE_FEATURES, MUON_FEATURES,
                     WANTED, check_registry, check_feature_map)

# aux_for eski l4_data surumlerinde yok.  Modul guncel degilse tikanmayalim:
# yerel bir esdegerine dus ve UYAR (asil cozum: git pull + Kernel Restart).
if hasattr(l4_data, "aux_for"):
    aux_for = l4_data.aux_for
else:
    print("[!] l4_data.aux_for YOK -> modul guncel degil.")
    print("    Dosyada var mi :", "def aux_for" in open(l4_data.__file__).read())
    print("    Modul yolu     :", l4_data.__file__)
    print("    Dosyada VAR ama burada YOK ise: Kernel > Restart")
    print("    Dosyada da YOK ise: git pull yapilmamis")
    _K = {"true_energy": ("signal",), "OneWeight": ("signal",),
          "NEvents": ("signal",), "pdg": ("signal",),
          "n_flux_events": ("signal",), "noise_weight": ("noise_bg",)}
    def aux_for(kind):
        return [k for k, v in _K.items() if kind in v]

print("--- noise BDT girdileri (Tablo 11) ---")
check_registry(TABLES, NOISE_FEATURES)
print("\n--- muon BDT girdileri (Tablo 12) ---")
check_registry(TABLES, MUON_FEATURES)

# AUX kolonlarinin hepsi HER ornekte yok: noise_weight sadece vuvuzela'da,
# OneWeight/PrimaryNeutrino* sadece GENIE'de.  Ornegin turune gore filtrele.
_kind = SAMPLES["nue"]["kind"]
print("\n--- agirlik kolonlari (%s ornegi) ---" % _kind)
check_registry(TABLES, aux_for(_kind))

print()
check_feature_map()

## 4. HDF5 → numpy

Tablolar `Run/Event/SubEvent` üzerinden eşleştiriliyor — satır sırasına
güvenilmiyor. Bir frame objesi bazı olaylarda yoksa sıraya dayalı okuma
**kayar** ve olay A'nın `cog_z`'si olay B'nin `NchCleaned`'i ile eşleşir.

Ağırlık böleni (`_n_files`) her HDF5'in yanındaki `.meta.json`'dan okunan
**L3 dosya sayısı**dır — HDF5 dosya sayısı değil.

In [ ]:
from l4_data import load_sample

data = {}
for name in SAMPLES:
    d = load_sample(name, SAMPLES, WANTED)
    if d is not None:
        data[name] = d

print("\nYuklendi:", {k: len(v["Run"]) for k, v in data.items()})

### Sağlık kontrolü

Bir değişken bir örnekte **tamamen** NaN'sa o kolon book edilmemiş demektir —
eğitime sokarsan model onu sessizce görmezden gelir.

In [ ]:
print("%-22s %s" % ("degisken", "  ".join("%9s" % s for s in data)))
for f in NOISE_FEATURES + MUON_FEATURES:
    row, bad = [], False
    for d in data.values():
        v = d.get(f)
        frac = 100.0 * np.mean(~np.isfinite(v)) if v is not None else 100.0
        bad |= frac > 99.9
        row.append("%8.1f%%" % frac)
    print("%-22s %s%s" % (f, "  ".join("%9s" % x for x in row),
                          "  <-- HEP EKSIK" if bad else ""))
print("\n(NaN yuzdesi.  %100 olan bir kolon book edilmemis demektir.)")

## 5. Ağırlıklar

Üç ayrı kavram:

1. **`w_phys` [Hz]** — fiziksel oran. Dağılım ve kesim performansı için.
2. **`w_train`** — eğitim ağırlığı (bölüm 6'da). Teknik notun ön işlemesi:
   sınıf toplamları eşitlenir, sonra 0–1 aralığına çekilir (§3.6.1).
3. **Ağırlıksız sayım** — istatistiksel yeterlilik.

`add_weights` sonucu teknik notun **Tablo 13** (L3 oranları) ile
karşılaştırıp mertebe sapmasını işaretler.

> `NORM`/`GAMMA` gerçek atmosferik akı **değil**, basit bir güç yasası. Mutlak
> oranlar birebir tutmaz; şekil karşılaştırması ve eğitim için yeterli.
> Gerçek akı için `nuflux` (Honda) + salınım gerekir.

In [ ]:
from l4_data import add_weights

add_weights(data)

In [ ]:
n = len(data)
fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 2.8))
axes = np.atleast_1d(axes)
for ax, (name, d) in zip(axes, data.items()):
    w = d["w_phys"]; w = w[np.isfinite(w) & (w > 0)]
    if w.size == 0:
        ax.set_title("%s: agirlik yok" % name, fontsize=8); continue
    ax.hist(np.log10(w), bins=40, color="tab:blue")
    ax.set_title("%s\nmaks/toplam = %.1f%%" % (name, 100 * w.max() / w.sum()),
                 fontsize=8)
    ax.set_xlabel("log10(w_phys)", fontsize=7); ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show()
print("maks/toplam > %5 ise tek bir olay orani domine ediyor.")

## 6. Eğitim setleri (`.npz`) ve train/test ayrımı

Her BDT için tek bir `.npz`: değişken kolonları + `label` (1 sinyal /
0 arka plan), `weight` (eğitim ağırlığı), `w_phys` (fiziksel ağırlık),
`istrain`, ve `features` (isim sırası — model ile dosya ayrışmasın diye).

**Eğitim ağırlığı burada hesaplanıyor**: sınıf toplamları eşitlenir, sonra
tüm ağırlıklar `[0, 1]`'e çekilir (teknik not §3.6.1).

İki ayrım tuzağı burada kapatılıyor:

1. **Sinyal ayrımı bir kez çekiliyor, iki BDT de aynısını kullanıyor.**
   Eskiden `make_datasets` her çağrıda yeniden çekiyordu; aynı olay noise
   BDT'sinin eğitim, muon BDT'sinin test setine düşebiliyordu. L4 kesimi
   ikisinin **birleşimi** olduğu için birleşik kesimi değerlendirecek ortak
   held-out set kalmıyordu → nihai verim olduğundan iyi görünürdü.
2. **CORSIKA duş (shower) bazında ayrılıyor, olay bazında değil.** Aynı hava
   duşu `OverSampling` kadar tekrar kullanılıyor; olay bazlı ayrım aynı duşun
   kopyalarını train ve test'e birden dağıtır → muon BDT'sinin test verimi
   şişer. Ayrım `Run` üzerinden yapılıyor. (Vuvuzela'da oversampling yok,
   gürültü olay bazında ayrılıyor.)


In [ ]:
TRAIN_FRAC = 0.5


def stack(samples, features):
    """Birden fazla ornegi tek bir {ad: dizi} sozluguna yig."""
    out = {f: np.concatenate([data[s][f] for s in samples]) for f in features}
    for extra in ("w_phys", "Run"):
        out[extra] = np.concatenate([data[s][extra] for s in samples])
    return out


def finite_mask(d, features, label):
    """
    Girdilerinden herhangi biri sonlu olmayan olaylari isaretle.

    LightGBM eksik degeri kendisi yonlendirir, yani ZORUNLU degil -- ama kac
    olayin hangi degisken yuzunden eksik oldugunu gormek istiyoruz.  %100 NaN
    bir degisken burada hemen goze carpar.
    """
    n = len(d["w_phys"])
    ok = np.ones(n, dtype=bool)
    for f in features:
        ok &= np.isfinite(np.asarray(d[f], dtype=np.float64))
    n_bad = int((~ok).sum())
    if n_bad:
        print("  [i] %-3s %d / %d olayda en az bir girdi NaN (%%%.2f)"
              % (label, n_bad, n, 100.0 * n_bad / n))
        for f in features:
            nf = int((~np.isfinite(np.asarray(d[f], dtype=np.float64))).sum())
            if nf:
                print("      %-22s %d" % (f, nf))
    return ok


def split_by_shower(runs, frac, rng_):
    """
    Duş (shower) bazinda train/test ayrimi.

    CORSIKA'da ayni hava dusu OverSampling kadar tekrarlanir; olay bazli
    ayrim ayni dusun kopyalarini iki tarafa birden dagitir ve test verimini
    sisirir.  Ayni `Run`'a ait olaylar hep birlikte hareket eder.
    """
    uniq = np.unique(runs)
    keep = set(uniq[rng_.random(len(uniq)) < frac].tolist())
    return np.array([r in keep for r in runs], dtype=bool)


def build_dataset(tag, sig, bg, features, sig_istrain, bg_istrain):
    """Bir BDT icin tek .npz yaz."""
    ws, wb = sig["w_phys"].copy(), bg["w_phys"].copy()
    for w in (ws, wb):
        w[~np.isfinite(w) | (w < 0)] = 0.0
    # sinif toplamlarini esitle, sonra [0,1]'e cek
    if ws.sum() > 0: ws /= ws.sum()
    if wb.sum() > 0: wb /= wb.sum()
    scale = max(ws.max(), wb.max())
    if scale > 0:
        ws /= scale; wb /= scale

    cols = {f: np.concatenate([np.asarray(sig[f], dtype=np.float64),
                               np.asarray(bg[f],  dtype=np.float64)])
            for f in features}
    cols["label"]   = np.r_[np.ones(len(ws)), np.zeros(len(wb))]
    cols["weight"]  = np.r_[ws, wb]
    cols["w_phys"]  = np.r_[np.asarray(sig["w_phys"], dtype=np.float64),
                            np.asarray(bg["w_phys"],  dtype=np.float64)]
    cols["istrain"] = np.r_[sig_istrain, bg_istrain]
    cols["features"] = np.array(features)

    path = os.path.join(DS_BASE, "L4_%s_dataset.npz" % tag)
    np.savez_compressed(path, **cols)
    print("  sinyal  train %7d / test %7d" % (sig_istrain.sum(), (~sig_istrain).sum()))
    print("  arkapln train %7d / test %7d" % (bg_istrain.sum(), (~bg_istrain).sum()))
    print("  -> %s" % path)
    return path


# --- sinyal BIR KEZ yigiliyor, ayrim BIR KEZ cekiliyor ---------------------
ALL_FEATURES = sorted(set(NOISE_FEATURES) | set(MUON_FEATURES))
SIG = stack(["nue", "numu"], ALL_FEATURES)
SIG_ISTRAIN = rng.random(len(SIG["w_phys"])) < TRAIN_FRAC
print("sinyal: %d olay, train %d / test %d  (iki BDT de AYNI ayrimi kullanir)"
      % (len(SIG_ISTRAIN), SIG_ISTRAIN.sum(), (~SIG_ISTRAIN).sum()))

In [ ]:
print("=== noise BDT  (sinyal = nue+numu, arkaplan = noise) ===")
BG_NOISE = stack(["noise"], ALL_FEATURES)
finite_mask(SIG, NOISE_FEATURES, "sig"); finite_mask(BG_NOISE, NOISE_FEATURES, "bg")
# vuvuzela'da oversampling yok -> olay bazli ayrim yeterli
BG_NOISE_ISTRAIN = rng.random(len(BG_NOISE["w_phys"])) < TRAIN_FRAC
DS_NOISE = build_dataset("noise", SIG, BG_NOISE, NOISE_FEATURES,
                         SIG_ISTRAIN, BG_NOISE_ISTRAIN)

print("\n=== muon BDT  (sinyal = nue+numu, arkaplan = corsika) ===")
BG_MUON = stack(["corsika"], ALL_FEATURES)
finite_mask(SIG, MUON_FEATURES, "sig"); finite_mask(BG_MUON, MUON_FEATURES, "bg")
# CORSIKA: DUS bazinda ayir (OverSampling sizintisi)
BG_MUON_ISTRAIN = split_by_shower(BG_MUON["Run"], TRAIN_FRAC, rng)
print("  corsika: %d benzersiz Run (dus), %d olay"
      % (len(np.unique(BG_MUON["Run"])), len(BG_MUON["Run"])))
DS_MUON = build_dataset("muon", SIG, BG_MUON, MUON_FEATURES,
                        SIG_ISTRAIN, BG_MUON_ISTRAIN)

## 7. Eğitim

`train_L4_classifier.py`'yi çağırır — eğitim mantığı tek yerde kalsın diye
burada tekrar yazılmıyor.

Hiperparametreler **teknik notun Tablo 10'u**, script içinde sabit. Elle
vermek gerekmiyor; sadece `--min-data-in-leaf` gibi tek tük şeyler
gerekirse komut satırından geçilir.

`--min-data-in-leaf` uyarısına dikkat: Tablo 10 bu değeri (500) referansın
çok daha büyük gürültü istatistiğine göre veriyor. Bizim örneğimizle
sonucu tek başına belirleyebilir; script bunu başta söylüyor.


In [ ]:
def run_train(tag, dataset, extra=()):
    cmd = [sys.executable, "-u", TRAIN_PY, "--tag", tag,
           "--dataset", dataset, "--outdir", MODEL_DIR] + list(extra)
    print("$ " + " ".join(shlex.quote(c) for c in cmd) + "\n")
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout)
    if p.returncode != 0:
        print("--- HATA ---\n" + (p.stderr or "")[-3000:])
    return p.returncode == 0


run_train("noise", DS_NOISE)

In [ ]:
run_train("muon", DS_MUON)

## 8. Doğrulama

`train_L4_classifier.py` her model için `.json` meta dosyası ve üç grafik
yazdı.

**Overtraining ölçütü train/test verim açığı** (`gap`), skor dağılımlarının
KS testi değil. Bu veri üzerinde ölçüldü: KS en iyi modeli "overtrained",
en kötüsünü "temiz" damgaladı — 165 bin sinyal olayıyla istatistiksel
olarak anlamlı ama fiziksel olarak önemsiz farkları yakalıyor. `gap` ise
doğrudan önemsediğimiz şeyi karşılaştırıyor: ezberleyen model train'de iyi,
test'te kötü olur.


In [ ]:
META = {}
for tag in ("noise", "muon"):
    p = os.path.join(MODEL_DIR, "L4_%s_model.json" % tag)
    if not os.path.exists(p):
        print("%-6s henuz egitilmedi" % tag)
        continue
    META[tag] = json.load(open(p))
    m = META[tag]["metrics"]
    gap = m.get("gap")
    print("%-6s %4d agac   %%%.0f redde verim %%%.1f  (arkaplan %s olay)   "
          "gap %s%s"
          % (tag, META[tag]["n_trees"],
             100 * META[tag]["target_rejection"],
             100 * (m["eff_at_target"] or float("nan")),
             m["bg_kept_at_target"],
             "%+.1f" % (100 * gap) if gap is not None else "?",
             "  <-- EZBERLIYOR" if (gap or 0) > 0.05 else ""))
    if (m["bg_kept_at_target"] or 0) < 10:
        print("       [!] hedef nokta 10'dan az arkaplan olayiyla tanimli "
              "-- olcum degil")

In [ ]:
from IPython.display import Image, display

# train_L4_classifier.py grafikleri <tag>_<kind>.png olarak yaziyor
for tag in META:
    for kind in ("cuts", "overtrain", "dist"):
        p = os.path.join(MODEL_DIR, "%s_%s.png" % (tag, kind))
        if os.path.exists(p):
            print(tag, kind)
            display(Image(filename=p))
        else:
            print("%s %s: yok (%s)" % (tag, kind, p))

In [ ]:
# Degisken onemi -- hangi degisken modeli tasiyor
for tag, meta in META.items():
    imp = meta["importance_gain"]
    tot = sum(imp.values()) or 1.0
    print("=== %s ===" % tag)
    for f, v in sorted(imp.items(), key=lambda x: -x[1]):
        print("  %-22s %8.1f  (%%%.1f)" % (f, v, 100 * v / tot))
    print()

## 9. Kesim seçimi

Model çıktısı LightGBM'in `P(sinyal)` olasılığı — teknik nottakiyle **aynı
ölçek**, yani v00.07'nin `noise ≥ 0.70` / `muon ≥ 0.65` değerleri doğrudan
kullanılabilir. `train_L4_classifier.py` zaten o kesimdeki verim/reddi
basıyor; aşağıdaki hücre kendi eğrinden seçmek istersen.

Referans hedefler (v00.07, pass2, Tablo 13):
- **noise**: 36.6 mHz → <0.3 mHz gürültü, nötrinoların ~%96'sı korunur
- **muon**: muonların %94'ü atılır, nötrinoların %87'si korunur

`.json`'daki `table` her red seviyesinde kesimi ve ham olay sayılarını
tutuyor — arka plan sayısı 10'un altındaki satırlar ölçüm değil.


In [ ]:
CUTS = {}
for tag, meta in META.items():
    print("=== %s ===  (not kesimi: %s)" % (tag, meta["default_cut"]))
    print("%-10s %9s %10s %12s" % ("red", "verim", "kesim", "arkaplan"))
    for row in meta["metrics"]["table"]:
        mark = "  <-- olcum degil" if row["bg_kept"] < 10 else ""
        print("%-9.1f%% %8.1f%% %10.4f %6d/%-6d%s"
              % (100 * row["rejection"], 100 * row["eff"], row["cut"],
                 row["bg_kept"], row["bg_total"], mark))
    CUTS[tag] = meta["default_cut"]
    print()

print("Kullanilacak kesimler:", CUTS)

## 10. Modeli frame'e uygulama

`.i3` dosyalarını yeniden işleyip her frame'e sınıflandırıcı skorunu yazmak
için `l4_classifier_module.py`:

```python
from l4_classifier_module import add_L4_classifiers

add_L4_classifiers(tray, "L4_clf", model_dir=MODEL_DIR,
                   noise_cut=0.70, muon_cut=0.65)
```

Bu iki modeli de ekler ve `noise ≥ 0.70 AND muon ≥ 0.65` birleşik L4
kesimini frame'e `I3Bool` olarak yazar.

> Modül değişkenleri frame'den `l4_classifier_module.FEATURE_MAP` üzerinden
> okur; `l4_data.REGISTRY` ise HDF5 kolonunu tarif eder. İkisi ayrışırsa
> eğitimde bir şey, uygulamada başka bir şey okunur ve model **hata
> vermeden** yanlış sonuç üretir. Aşağıdaki kontrol bunu yakalar (bölüm
> 3'tekiyle aynı).


In [ ]:
check_feature_map()

## Kontrol listesi

**İşleme**
- [ ] Smoke test temiz, kademe makul (bölüm 1)
- [ ] `check_registry` tüm BDT girdilerini buldu (bölüm 3)
- [ ] `check_feature_map()` çakışma bulmadı (bölüm 3)
- [ ] Hiçbir değişken %100 NaN değil (bölüm 4)

**Ağırlıklar**
- [ ] `.meta.json` uyarısı çıkmadı (yoksa bölen yanlış → oranlar kayar)
- [ ] `maks/toplam < %5`
- [ ] Tablo 13 ile mertebe tutuyor (bölüm 5)

**Eğitim**
- [ ] `gap < 0.05` — ezberleme yok (bölüm 8)
- [ ] Hedef noktayı ≥10 arka plan olayı tanımlıyor (bölüm 8)
- [ ] `min_data_in_leaf` uyarısı çıktıysa dikkate alındı (bölüm 7)
- [ ] Kesim seçildi, verim/red referans mertebelerde (bölüm 9)

---

**Hâlâ doğrulanmamış olanlar** — ayrıntı `CLAUDE.md` → *Açık riskler* ve
`TEKNIK_NOT_KARSILASTIRMA.md`:

- VICH COG'unun yük ağırlıklı olup olmadığı
- `accumulated_time`'ın referans zamanı (ilk pulse mü, tetikleme mi)
- `fill_ratio`'nun `SphericalRadiusMean=1.6` parametresi oscNext için
  yeniden optimize edilmedi — ve gürültü modelinin gain'inin %61'ini bu
  değişken taşıyor, yani en yüksek getirili tek ayar
- Muon BDT arka planı gerçek veri yerine CORSIKA → data/MC kontrolü yok
- ντ seti yok → sinyalin ~%3'ü eksik
- Gürültü MC istatistiği düşük (~2000 olay): %99'un sağı ölçülemiyor
